# Data Profiling e Qualidade — `olist_orders_dataset`

Duas etapas sobre a tabela-fato central do dataset Olist:

1. **Profiling exploratório** com `ydata-profiling` — o que existe nos dados.
2. **Avaliação de qualidade** com `pandera` — se os dados atendem às regras esperadas,
   organizada por dimensões de qualidade.

## Ambiente

O `ydata-profiling` não instala no Python 3.14 do projeto: depende de `numba`/`llvmlite`, que ainda
não publicam wheels para o CPython 3.14 e falham ao compilar do fonte. Por isso existe um ambiente
isolado, que não altera o `.venv` principal:

```bash
uv venv .venv-profiling --python 3.12
uv pip install --python .venv-profiling ydata-profiling pandera ipykernel pandas "setuptools<81"
.venv-profiling/Scripts/python.exe -m ipykernel install --user \
    --name olist-profiling --display-name "Python 3.12 (olist-profiling)"
```

**Selecione o kernel `Python 3.12 (olist-profiling)` antes de executar.** O pin `setuptools<81` é
necessário porque o `ydata-profiling` 4.18.4 ainda importa `pkg_resources`, removido no setuptools 81.

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd

try:
    import pandera
    import pandera.pandas as pa  # namespace recomendado a partir do pandera 0.24
    import ydata_profiling
    from pandera.pandas import Check, Column, DataFrameSchema
    from ydata_profiling import ProfileReport
except ModuleNotFoundError as erro:
    raise ModuleNotFoundError(
        f"'{erro.name}' não encontrado. Selecione o kernel 'Python 3.12 (olist-profiling)' —\n"
        "o ydata-profiling não instala no Python 3.14 do projeto (numba/llvmlite sem wheel)."
    ) from erro

warnings.filterwarnings("ignore")

print(f"Python          : {sys.version.split()[0]}")
print(f"pandas          : {pd.__version__}")
print(f"ydata-profiling : {ydata_profiling.__version__}")
print(f"pandera         : {pandera.__version__}")

In [ ]:
# O notebook roda a partir de notebooks/, então a raiz do projeto é o diretório pai
PROJ_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJ_ROOT / "data" / "raw"
OUT_DIR = PROJ_ROOT / "reports" / "profiling"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TABELA = "olist_orders_dataset"
ARQUIVO = RAW_DIR / f"{TABELA}.csv"

if not ARQUIVO.exists():
    raise FileNotFoundError(f"{ARQUIVO} não existe. Confirme que os CSVs estão em data/raw/.")

print(f"entrada : {ARQUIVO}")
print(f"saida   : {OUT_DIR}")

## 1. Carga

As 5 colunas de data são convertidas já no `read_csv` — sem isso o ydata as trata como texto
categórico de altíssima cardinalidade e perde toda a análise temporal, e os checks de ordenação
temporal do pandera comparariam strings.

In [ ]:
DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders = pd.read_csv(ARQUIVO, parse_dates=DATE_COLS)

print(f"{orders.shape[0]:,} linhas x {orders.shape[1]} colunas\n")
orders.info()

In [ ]:
orders.head()

# Parte 1 — Profiling exploratório (ydata-profiling)

Três escolhas de configuração:

- **`explorative=True`** — com 99 mil linhas o modo completo roda em tempo aceitável e traz
  correlações e o heatmap de faltantes, que é justamente onde está o interesse aqui: os nulos
  das datas de entrega não são aleatórios, dependem do `order_status`.
- **`order_id` e `customer_id` como categóricas** — são hashes de 32 caracteres com cardinalidade
  igual ao número de linhas. Marcá-las evita que o ydata tente inferir texto e gere ruído.
- **`minify_html=False`** — a minificação depende do pacote opcional `minify_html`, que não faz
  parte da instalação. Desligar evita a dependência; o relatório sai idêntico, só maior.

In [ ]:
perfil = ProfileReport(
    orders,
    title=f"Profiling — {TABELA}",
    explorative=True,
    missing_diagrams={"bar": True, "matrix": True, "heatmap": True},
    type_schema={"order_id": "categorical", "customer_id": "categorical"},
    html={"minify_html": False},
)

destino = OUT_DIR / f"{TABELA}.html"
perfil.to_file(destino)
print(f"\nSalvo em: {destino}  ({destino.stat().st_size / 1024**2:.1f} MB)")

In [ ]:
perfil.to_notebook_iframe()

### Alertas emitidos pelo ydata

In [ ]:
alertas = pd.DataFrame(
    [
        {"tipo": a.alert_type.name, "coluna": a.column_name}
        for a in perfil.get_description().alerts
    ]
)

print(f"{len(alertas)} alertas\n")
alertas

# Parte 2 — Qualidade de dados (pandera)

O profiling descreve o que existe; ele não diz se está **certo**. Para isso as regras precisam ser
declaradas explicitamente e verificadas. É o que o `pandera` faz — e o resultado é organizado aqui
nas dimensões clássicas de qualidade de dados:

| Dimensão | Pergunta que responde | Como é medida aqui |
|---|---|---|
| **Completude** | Os dados obrigatórios estão presentes? | Nulos em chaves e datas obrigatórias, e nulos *condicionais* (pedido entregue precisa ter data de entrega) |
| **Unicidade** | Há duplicatas onde deveria haver chave? | `order_id` e `customer_id` sem repetição |
| **Validade** | Os valores respeitam formato e domínio? | Hash hex de 32 chars, `order_status` no domínio conhecido, datas dentro da janela do dataset |
| **Consistência** | Os campos fazem sentido entre si? | Ordenação temporal: compra → aprovação → postagem → entrega |
| **Acurácia** | Os valores são plausíveis no mundo real? | Lead time de entrega dentro de faixa razoável |
| **Atualidade** | Os prazos e datas são coerentes no tempo? | Prazo prometido a uma distância plausível da compra |

**Nota sobre acurácia:** sem uma fonte de verdade externa não há como medir acurácia de fato. O que
se mede aqui é *plausibilidade* — um lead time de 200 dias não prova erro, mas é implausível o
bastante para exigir investigação. É a melhor aproximação possível com o dado disponível.

### Sobre a escolha de checks explícitos

O pandera oferece `nullable=False` e `unique=True` direto na `Column`. Aqui as duas regras viram
checks nomeados (`Completude :: order_id preenchido`) porque o nome do check é o que carrega a
dimensão até o scorecard. Os atalhos implícitos reportariam tudo como `not_nullable`, colapsando
todas as colunas numa linha só e perdendo a granularidade.

In [ ]:
STATUS_VALIDOS = [
    "delivered",
    "shipped",
    "canceled",
    "unavailable",
    "invoiced",
    "processing",
    "created",
    "approved",
]
JANELA = (pd.Timestamp("2016-09-01"), pd.Timestamp("2018-11-01"))
LIMITE_DIAS = 180

# Cada check criado por dim() se registra aqui, para o scorecard incluir tambem os que passaram
REGISTRO = []


def dim(dimensao: str, descricao: str, fn, **kw) -> Check:
    """Cria um Check nomeado 'Dimensao :: descricao' e o registra para o scorecard."""
    nome = f"{dimensao} :: {descricao}"
    REGISTRO.append({"dimensao": dimensao, "check": nome})
    return Check(fn, name=nome, **kw)


def ordem_temporal(df, antes, depois):
    """antes <= depois onde ambas as datas existem; linha sem dado nao conta como violacao."""
    avaliavel = df[antes].notna() & df[depois].notna()
    return ~avaliavel | (df[antes] <= df[depois])


def dias(df, inicio, fim):
    return (df[fim] - df[inicio]).dt.total_seconds() / 86400

In [ ]:
schema = DataFrameSchema(
    name="olist_orders",
    strict=True,  # colunas fora do schema tambem sao violacao (dimensao Estrutura)
    columns={
        "order_id": Column(
            str,
            [
                dim("Completude", "order_id preenchido", lambda s: s.notna()),
                dim("Unicidade", "order_id sem duplicatas", lambda s: ~s.duplicated(keep=False)),
                dim(
                    "Validade",
                    "order_id e hash hex de 32 chars",
                    lambda s: s.str.fullmatch(r"[0-9a-f]{32}"),
                ),
            ],
            nullable=True,
        ),
        "customer_id": Column(
            str,
            [
                dim("Completude", "customer_id preenchido", lambda s: s.notna()),
                dim(
                    "Unicidade",
                    "customer_id sem duplicatas",
                    lambda s: ~s.duplicated(keep=False),
                ),
                dim(
                    "Validade",
                    "customer_id e hash hex de 32 chars",
                    lambda s: s.str.fullmatch(r"[0-9a-f]{32}"),
                ),
            ],
            nullable=True,
        ),
        "order_status": Column(
            str,
            [
                dim("Completude", "order_status preenchido", lambda s: s.notna()),
                dim(
                    "Validade",
                    "order_status no dominio conhecido",
                    lambda s: s.isin(STATUS_VALIDOS),
                ),
            ],
            nullable=True,
        ),
        "order_purchase_timestamp": Column(
            "datetime64[ns]",
            [
                dim("Completude", "data de compra preenchida", lambda s: s.notna()),
                dim(
                    "Validade",
                    "compra dentro da janela do dataset",
                    lambda s: s.between(*JANELA),
                ),
            ],
            nullable=True,
        ),
        # Nulos aqui sao esperados: dependem do estagio do pedido
        "order_approved_at": Column("datetime64[ns]", nullable=True),
        "order_delivered_carrier_date": Column("datetime64[ns]", nullable=True),
        "order_delivered_customer_date": Column("datetime64[ns]", nullable=True),
        "order_estimated_delivery_date": Column(
            "datetime64[ns]",
            [dim("Completude", "prazo prometido preenchido", lambda s: s.notna())],
            nullable=True,
        ),
    },
    checks=[
        # Consistencia: a linha do tempo do pedido tem que ser monotonica
        dim(
            "Consistencia",
            "compra <= aprovacao",
            lambda df: ordem_temporal(df, "order_purchase_timestamp", "order_approved_at"),
        ),
        dim(
            "Consistencia",
            "aprovacao <= postagem",
            lambda df: ordem_temporal(df, "order_approved_at", "order_delivered_carrier_date"),
        ),
        dim(
            "Consistencia",
            "postagem <= entrega",
            lambda df: ordem_temporal(
                df, "order_delivered_carrier_date", "order_delivered_customer_date"
            ),
        ),
        dim(
            "Consistencia",
            "compra <= prazo prometido",
            lambda df: ordem_temporal(
                df, "order_purchase_timestamp", "order_estimated_delivery_date"
            ),
        ),
        # Completude condicional: o status implica quais datas precisam existir
        dim(
            "Completude",
            "pedido entregue tem data de entrega",
            lambda df: ~(
                (df.order_status == "delivered") & df.order_delivered_customer_date.isna()
            ),
        ),
        dim(
            "Completude",
            "pedido entregue tem data de postagem",
            lambda df: ~(
                (df.order_status == "delivered") & df.order_delivered_carrier_date.isna()
            ),
        ),
        # Plausibilidade
        dim(
            "Acuracia",
            f"lead time de entrega entre 0 e {LIMITE_DIAS} dias",
            lambda df: df.order_delivered_customer_date.isna()
            | dias(df, "order_purchase_timestamp", "order_delivered_customer_date").between(
                0, LIMITE_DIAS
            ),
        ),
        dim(
            "Atualidade",
            f"prazo prometido ate {LIMITE_DIAS} dias apos a compra",
            lambda df: dias(
                df, "order_purchase_timestamp", "order_estimated_delivery_date"
            ).between(0, LIMITE_DIAS),
        ),
    ],
)

print(f"{len(REGISTRO)} checks declarados em {pd.Series([r['dimensao'] for r in REGISTRO]).nunique()} dimensoes")

### Validação

`lazy=True` é essencial: sem ele o pandera aborta na primeira violação e só se enxerga um problema
por execução. Com ele, todos os checks rodam e as falhas voltam consolidadas em `failure_cases`.

In [ ]:
COLS_FALHA = ["schema_context", "column", "check", "failure_case", "index"]

try:
    schema.validate(orders, lazy=True)
    falhas = pd.DataFrame(columns=COLS_FALHA)
    print("Nenhuma violacao encontrada.")
except pa.errors.SchemaErrors as erro:
    falhas = erro.failure_cases
    print(f"{len(falhas):,} casos de falha registrados")

falhas.head()

### Scorecard

O `failure_cases` só lista o que falhou. O `REGISTRO` preenchido por `dim()` permite fazer o
caminho inverso e mostrar também os checks que passaram — sem isso o scorecard daria a impressão
de que só existem 5 regras, quando na verdade 19 foram avaliadas.

In [ ]:
TOTAL = len(orders)

violacoes = (
    falhas.groupby("check").agg(linhas_violando=("index", "nunique"))
    if len(falhas)
    else pd.DataFrame(columns=["linhas_violando"])
)

scorecard = (
    pd.DataFrame(REGISTRO)
    .merge(violacoes, on="check", how="left")
    .fillna({"linhas_violando": 0})
    .astype({"linhas_violando": int})
    .assign(conformidade_pct=lambda d: (1 - d.linhas_violando / TOTAL) * 100)
    .sort_values(["dimensao", "linhas_violando"], ascending=[True, False])
    .reset_index(drop=True)
)

scorecard.to_csv(OUT_DIR / f"qualidade_{TABELA}.csv", index=False)
scorecard.round(3)

In [ ]:
# Conformidade da dimensao = a do seu pior check (a regra mais violada define o gargalo)
por_dimensao = (
    scorecard.groupby("dimensao")
    .agg(
        checks=("check", "count"),
        com_falha=("linhas_violando", lambda s: int((s > 0).sum())),
        pior_check_linhas=("linhas_violando", "max"),
    )
    .assign(conformidade_pct=lambda d: (1 - d.pior_check_linhas / TOTAL) * 100)
    .sort_values("conformidade_pct")
)

print(f"{TOTAL:,} linhas avaliadas | {len(scorecard)} checks | "
      f"{int((scorecard.linhas_violando > 0).sum())} com falha\n")
por_dimensao.round(3)

### Investigação da maior violação

A dimensão mais fraca é **Consistência**, puxada pelo check `aprovacao <= postagem`. Vale entender
se é erro de registro ou prática operacional antes de tratar como sujeira.

In [ ]:
postado_antes = orders[
    orders.order_approved_at.notna()
    & orders.order_delivered_carrier_date.notna()
    & (orders.order_delivered_carrier_date < orders.order_approved_at)
].copy()

postado_antes["horas_de_antecedencia"] = (
    postado_antes.order_approved_at - postado_antes.order_delivered_carrier_date
).dt.total_seconds() / 3600

print(f"Pedidos postados antes da aprovacao do pagamento: {len(postado_antes):,} "
      f"({len(postado_antes) / TOTAL:.2%})\n")
print("Antecedencia em horas:")
print(postado_antes.horas_de_antecedencia.describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string())
print("\nStatus desses pedidos:")
print(postado_antes.order_status.value_counts().to_string())

In [ ]:
# As demais violacoes, para fechar o quadro
entregue_antes = orders[
    orders.order_delivered_customer_date < orders.order_delivered_carrier_date
]
print(f"Entregues antes de sair para a transportadora : {len(entregue_antes)}")

lead = (orders.order_delivered_customer_date - orders.order_purchase_timestamp).dt.days
print(f"Lead time acima de {LIMITE_DIAS} dias                     : {int((lead > LIMITE_DIAS).sum())}")

entregue_sem_data = orders[
    (orders.order_status == "delivered") & orders.order_delivered_customer_date.isna()
]
print(f"Status 'delivered' sem data de entrega        : {len(entregue_sem_data)}")

entregue_sem_postagem = orders[
    (orders.order_status == "delivered") & orders.order_delivered_carrier_date.isna()
]
print(f"Status 'delivered' sem data de postagem       : {len(entregue_sem_postagem)}")

### Faltantes por status

Fecha o argumento da dimensão Completude: a maior parte dos nulos das datas não é defeito, é
consequência do estágio do pedido. Um pedido `shipped` não tem data de entrega porque ainda não
foi entregue — e é por isso que os checks de completude aqui são condicionais ao status, e não
um simples `nullable=False`.

In [ ]:
COLS_NULAS = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]

faltantes = (
    orders.assign(**{c: orders[c].isna() for c in COLS_NULAS})
    .groupby("order_status")[COLS_NULAS]
    .sum()
    .assign(total_pedidos=orders.order_status.value_counts())
    .sort_values("total_pedidos", ascending=False)
)

faltantes